# ECON 4370 / BANA 4373 — Homework 5 Starter Notebook
## A/B Testing: From Design to Decision

**Posted:** Apr. 14  
**Due:** Monday, Apr. 20 (11:59 PM, Blackboard Ultra)

> **How to use this notebook:**
> - Replace every **TODO** with your answer (code or written response).
> - Keep written responses in **Markdown** cells. Keep code in **Code** cells.
> - The notebook must run **top → bottom without errors** before you submit.
> - Save as `.ipynb` and upload to Blackboard Ultra.

---


## Student Information
- **Name:** TODO
- **Section / Time:** TODO
- **NetID / Email:** TODO (optional)


---
## 0) Setup — Imports and Folder Structure


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
import os

# Reproducibility
np.random.seed(4373)

# Folder structure (mirrors course convention)
for folder in ['data_raw', 'data_clean', 'exports']:
    os.makedirs(folder, exist_ok=True)

print('Setup complete.')


---
# Part 1: The A/B Testing Framework (No Code)

Answer the following questions in your own words. Keep each answer concise
(2–4 sentences unless otherwise stated).


## Question 1 — Why Randomization Solves Selection Bias  *(4 points)*

Recall the **Selection bias decomposition** from the potential outcomes lecture:

$$\text{Naive estimate} = \underbrace{\text{ATT}}_{\text{what we want}} +
\underbrace{E[Y_i^0 | D_i=1] - E[Y_i^0 | D_i=0]}_{\text{selection bias}}$$

**(a)** Explain in your own words what the selection bias term represents.
Why does it equal zero when treatment is randomly assigned?

**(b)** A company notices that customers who *voluntarily* sign up for their
loyalty program spend 40% more per year than non-members. A manager says:
*"The loyalty program causes customers to spend more — we should expand it."*
What is wrong with this reasoning? What type of bias is present?


**Answer (a):** TODO

**Answer (b):** TODO


## Question 2 — The 2×2 Table  *(5 points)*

A streaming service ran an A/B test of a new recommendation algorithm.
The table below shows the **average daily watch time (minutes)** for
Control and Treatment users, measured the week before and the week after launch.

| | **Pre-launch week** | **Post-launch week** |
|---|---|---|
| **Treatment (new algorithm)** | 42.0 | 51.0 |
| **Control (old algorithm)** | 40.0 | 45.0 |

**(a)** What is the simple **before–after** estimate for the treatment group?

**(b)** What is the control group's change over the same period?

**(c)** Compute the **difference-in-differences** estimate.

**(d)** The service also collected a **cross-sectional** estimate: they compared
treatment vs. control only in the post-launch week. Compute it.

**(e)** In a well-randomized A/B test, should (c) and (d) give similar answers?
Why? Under what condition would they differ?


**Answer (a):** TODO

**Answer (b):** TODO

**Answer (c):** TODO

**Answer (d):** TODO

**Answer (e):** TODO


## Question 3 — Hypothesis Testing Concepts  *(5 points)*

**(a)** Define Type I error and Type II error in the context of an A/B test.
Give a concrete business example of each.

**(b)** A test returns $p = 0.12$. A colleague says: *"There is a 12% chance
the new feature has no effect."* What is wrong with this interpretation?
What does $p = 0.12$ actually mean?

**(c)** Explain why a confidence interval is more informative than a p-value
alone for a business decision.


**Answer (a):** TODO

**Answer (b):** TODO

**Answer (c):** TODO


## Question 4 — Statistical vs. Economic Significance  *(4 points)*

A large e-commerce platform runs an A/B test with $n = 2{,}000{,}000$ users
per group. They find that the new checkout button increases conversion
by **0.003 percentage points** (e.g., from 20.000% to 20.003%).
The result is statistically significant at $p < 0.001$.

**(a)** Why does a very large sample size make it easier to detect tiny effects?
Use the standard error formula to explain.

**(b)** The platform ships 5 million orders per month. Is a 0.003pp lift
economically meaningful if the average order value is \$80?
Compute the estimated monthly revenue impact.

**(c)** What does this example teach us about always relying on p-values alone
to make business decisions?


**Answer (a):** TODO

**Answer (b):** TODO

**Answer (c):** TODO


---
# Part 2: Simulating an A/B Test with a Known True Effect

You will now generate a **calibrated synthetic dataset** where the true lift
is known. This lets you check whether your estimator recovers it — something
you can never do with real data.

### Context

A music streaming service is testing a **new playlist generation algorithm**
(Treatment) against the existing one (Control). The outcome is whether a user
**completes a listening session** without skipping away — coded as
`y = 1` (completed) or `y = 0` (dropped off).

**Design parameters (do not change):**
- `N_per_group = 500` — 500 users in each group
- `BASELINE_RATE = 0.35` — 35% of control users complete a session
- `TRUE_LIFT = 0.06` — the new algorithm truly lifts completion by 6pp
- `ALPHA = 0.05` — significance level


## Question 5 — Write Your Prior  *(3 points)*

**Before running any code**, answer the following:

**(a)** With $n = 500$ per group and a true lift of 6pp, do you expect the
experiment to be statistically significant at the 5% level? Why or why not?
(You will verify this formally in Question 7.)

**(b)** If the true lift were only 1pp instead of 6pp, would your answer change?
What does this tell you about the importance of power analysis?


**Prior (a):** TODO

**Prior (b):** TODO


## Question 6 — Simulate the Experiment  *(4 points)*


In [ ]:
# ── Design parameters ──────────────────────────────────────────────────────
N_per_group   = 500
BASELINE_RATE = 0.35    # control group completion rate
TRUE_LIFT     = 0.06    # the number we want to recover
ALPHA         = 0.05

rng = np.random.default_rng(4373)

# ── Generate outcomes ────────────────────────────────────────────────────────
# Control group: Bernoulli(BASELINE_RATE)
y_control   = rng.binomial(1, BASELINE_RATE, N_per_group)

# Treatment group: Bernoulli(BASELINE_RATE + TRUE_LIFT)
y_treatment = rng.binomial(1, BASELINE_RATE + TRUE_LIFT, N_per_group)

# ── Build a tidy dataframe ───────────────────────────────────────────────────
df = pd.DataFrame({
    'user_id' : np.arange(2 * N_per_group),
    'treat'   : np.r_[np.zeros(N_per_group, dtype=int),
                      np.ones(N_per_group,  dtype=int)],
    'y'       : np.r_[y_control, y_treatment],
})
df['group'] = df['treat'].map({0: 'Control', 1: 'Treatment'})

# Save raw data
df.to_csv('data_raw/ab_test_raw.csv', index=False)

print(f'Rows: {len(df)}')
print(f'Control users: {(df.treat==0).sum()}')
print(f'Treatment users: {(df.treat==1).sum()}')
df.head(8)


**Describe the dataset (2–3 sentences):** What does each row represent?
What does `y = 1` mean in this business context?
Why does having a fixed `TRUE_LIFT` help us evaluate our estimator?


**Answer:** TODO


---
# Part 3: Estimating the Treatment Effect (Manual)


## Question 7 — Summary Statistics and the ATE Estimate  *(10 points)*


In [ ]:
# TODO: Compute group-level summary statistics
# For each group compute: n, number of completions, completion rate
# Store in variables: n_c, n_t, complete_c, complete_t, rate_c, rate_t

# YOUR CODE HERE


# TODO: Compute the ATE estimate (difference in means)
tau_hat = # TODO

# TODO: Compute the standard error
# SE = sqrt( rate_t*(1-rate_t)/n_t  +  rate_c*(1-rate_c)/n_c )
se = # TODO

# TODO: Compute the 95% confidence interval
z_crit  = stats.norm.ppf(1 - ALPHA / 2)   # 1.96
ci_low  = # TODO
ci_high = # TODO

# TODO: Compute the z-statistic and two-tailed p-value
z_stat  = # TODO
p_value = # TODO

print(f'Control completion rate :  {rate_c:.3%}  (n={n_c})')
print(f'Treatment completion rate: {rate_t:.3%}  (n={n_t})')
print(f'Estimated lift (tau_hat) : {tau_hat:.3%}')
print(f'True lift (known)        : {TRUE_LIFT:.3%}')
print(f'Standard error           : {se:.3%}')
print(f'95% CI                   : [{ci_low:.3%}, {ci_high:.3%}]')
print(f'z-statistic              : {z_stat:.3f}')
print(f'p-value                  : {p_value:.4f}')
print(f'Decision: {"Reject H0" if p_value < ALPHA else "Fail to reject H0"}')


**Interpret your results (3–4 sentences):**
Is the estimated lift statistically significant? Does it match the true lift?
Was your prior from Question 5 correct? Explain any discrepancy between
`tau_hat` and `TRUE_LIFT`.


**Answer:** TODO


## Question 8 — Visualize the Result  *(7 points)*

Create a **two-panel figure**:
- **Left panel**: bar chart of completion rates for Control and Treatment,
  with 95% CI error bars around **each group mean** separately.
- **Right panel**: a dot-and-whisker plot showing the **estimated lift**
  (tau_hat) with its 95% CI, plus a vertical dashed line at zero.

Save the figure to `exports/ab_result.png`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ── Left panel: completion rates ─────────────────────────────────────────────
ax = axes[0]
# TODO: compute per-group CI half-widths and plot bars with error bars
# ci_half_c = z_crit * sqrt(rate_c*(1-rate_c)/n_c)
# ci_half_t = z_crit * sqrt(rate_t*(1-rate_t)/n_t)

# YOUR CODE HERE

ax.set_ylabel('Completion rate')
ax.set_title('Completion rate by group\n(error bars = 95% CI per group)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# ── Right panel: CI for the difference ───────────────────────────────────────
ax2 = axes[1]
# TODO: plot tau_hat as a dot with horizontal error bars (xerr = z_crit * se)
# Add a vertical dashed line at x=0 ("no effect")
# Label the x-axis and add a title

# YOUR CODE HERE

plt.tight_layout()
plt.savefig('exports/ab_result.png', dpi=150, bbox_inches='tight')
plt.show()


**Describe your figure (2–3 sentences):**
Does the right panel's CI include zero or exclude it?
What does that tell you about the hypothesis test? Which panel should you
show to a manager who asks 'did the new algorithm work?'


**Answer:** TODO


---
# Part 4: OLS Regression and the ANCOVA Extension

The difference-in-means estimator is algebraically identical to the OLS
coefficient on the treatment indicator. This section asks you to verify
that relationship and then exploit it to improve precision.


## Question 9 — Write Your Prior for β₁  *(2 points)*

You are about to run the regression `y ~ treat`.

Before running it, write down:
**(a)** What value do you expect for the **intercept** and why?
**(b)** What value do you expect for the **treat coefficient** and why?


**Prior (a):** TODO

**Prior (b):** TODO


## Question 10 — Run the OLS Regression  *(10 points)*


In [ ]:
# TODO: Run OLS regression of y on treat
# Use statsmodels formula API: smf.ols('y ~ treat', data=df).fit()

model_simple = # TODO

intercept  = model_simple.params['Intercept']
treat_coef = model_simple.params['treat']

print('OLS: y_i = alpha + tau * treat_i + epsilon_i')
print('='*55)
print(f'Intercept (alpha)     = {intercept:.6f}')
print(f'treat coef (tau)      = {treat_coef:.6f}')
print()
print(f'Control rate (direct) = {rate_c:.6f}   <- should match intercept')
print(f'tau_hat (direct)      = {tau_hat:.6f}   <- should match treat coef')
print()
print(f'Intercept == Control rate? {abs(intercept - rate_c) < 1e-10}')
print(f'treat coef == tau_hat?     {abs(treat_coef - tau_hat) < 1e-10}')
print()
print(model_simple.summary().tables[1])


**Question 10a — Coefficient interpretation:**
In plain English, interpret both coefficients ($\hat\alpha$ and $\hat\tau$)
in the business context of this experiment.
What does each one represent?


**Intercept (alpha):** TODO

**treat coef (tau):** TODO


**Question 10b:** Does $\hat\tau$ from OLS match `tau_hat` from Question 7?
Explain *why* OLS on a binary treatment indicator always gives the same
result as the difference in means. (Hint: think about what the fitted model
predicts when `treat = 0` and when `treat = 1`.)


**Answer:** TODO


## Question 11 — ANCOVA: Adding a Pre-Treatment Covariate  *(7 points)*

Now add a **pre-treatment covariate**: the number of sessions the user
completed in the **previous week** (before the experiment started).
This variable — `prev_sessions` — is correlated with the outcome but
is independent of treatment assignment (because randomization happened
after it was measured).

Adding it to the regression is called **ANCOVA** (Analysis of Covariance).
It should **not** change the estimate of $\hat\tau$ (randomization took care
of confounding) but it **should** reduce the standard error.


In [ ]:
# ── Generate a pre-treatment covariate ─────────────────────────────────────
# prev_sessions: previous week's completed sessions per user.
# It is correlated with y (engaged users tend to stay engaged)
# but independent of treat (measured before randomization).
rng2 = np.random.default_rng(99)
df['prev_sessions'] = rng2.poisson(lam=4, size=len(df))   # avg 4 sessions/week

# TODO: Run ANCOVA — regress y on treat AND prev_sessions
model_ancova = # TODO

print('ANCOVA: y ~ treat + prev_sessions')
print('='*55)
print(model_ancova.summary().tables[1])
print()
print(f'Simple OLS  — treat coef: {model_simple.params["treat"]:.6f}',
      f'  SE: {model_simple.bse["treat"]:.6f}')
print(f'ANCOVA      — treat coef: {model_ancova.params["treat"]:.6f}',
      f'  SE: {model_ancova.bse["treat"]:.6f}')
print()
# TODO: compute and print the % reduction in SE
se_reduction = # TODO: (simple_SE - ancova_SE) / simple_SE * 100
print(f'SE reduction from adding covariate: {se_reduction:.1f}%')


**Interpret (3–4 sentences):**
Did the estimate of $\hat\tau$ change when you added `prev_sessions`?
Did the standard error change? In which direction, and why?
What does this imply for experiment design in practice?


**Answer:** TODO


---
# Part 5: Power Analysis and Sample Size

You must determine sample sizes **before** running an experiment,
not after. This section asks you to compute the MDE for various sample
sizes and work backwards to determine the sample size needed for
a specific MDE.

The **MDE formula** for a two-sample test of proportions:

$$n \;\geq\;
\frac{(z_{1-\alpha/2} + z_{1-\beta})^2 \;\cdot\; 2\sigma^2}{\delta^2}$$

Rearranged to give the MDE given $n$:

$$\text{MDE} = (z_{1-\alpha/2} + z_{1-\beta}) \cdot \sigma \cdot \sqrt{\frac{2}{n}}$$

where $\sigma = \sqrt{p(1-p)}$ is the standard deviation of the binary outcome
(use $p = \text{BASELINE\_RATE} = 0.35$) and the default settings are
$\alpha = 0.05$, power $= 0.80$.


## Question 12 — Write Your Prior for the MDE  *(2 points)*

Before computing anything:

**(a)** Our experiment used $n = 500$ per group. Do you expect the MDE
to be larger or smaller than the true lift of 6pp? Explain.

**(b)** If you doubled the sample size to $n = 1{,}000$ per group,
would the MDE halve? Why or why not?


**Prior (a):** TODO

**Prior (b):** TODO


## Question 13 — MDE Calculation and Power Curve  *(8 points)*


In [ ]:
# ── Helper: MDE given n ─────────────────────────────────────────────────────
def compute_mde(n_per_group, baseline=0.35, alpha=0.05, power=0.80):
    sigma   = np.sqrt(baseline * (1 - baseline))
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_power = stats.norm.ppf(power)
    return (z_alpha + z_power) * sigma * np.sqrt(2 / n_per_group)

# ── Part A: MDE for today's experiment ──────────────────────────────────────
mde_today = compute_mde(N_per_group)
print(f'MDE for n = {N_per_group} per group : {mde_today:.2%}')
print(f'True lift                : {TRUE_LIFT:.2%}')
print(f'Were we well-powered?    : {"YES" if TRUE_LIFT >= mde_today else "NO — underpowered"}')
print()

# ── Part B: MDE table ────────────────────────────────────────────────────────
print(f'{"n per group":>12}  {"MDE":>10}  {"Detects 6pp lift?":>20}')
print('-' * 48)
for n in [100, 250, 500, 1000, 2500, 5000]:
    mde = compute_mde(n)
    flag = 'YES' if 0.06 >= mde else 'No'
    print(f'{n:>12}  {mde:>10.2%}  {flag:>20}')


In [ ]:
# ── Part C: Power curve plot ─────────────────────────────────────────────────
# TODO: Plot MDE (in pp) on the y-axis against sample size per group (100 to 5000)
# on the x-axis. Add:
#   - A horizontal dashed line at 6pp (the true lift)
#   - A vertical dotted line at n=500 (today's experiment)
#   - Axis labels, title, and legend
# Save to exports/power_curve.png

sample_sizes = np.arange(100, 5100, 50)

# YOUR CODE HERE

plt.savefig('exports/power_curve.png', dpi=150, bbox_inches='tight')
plt.show()


**Interpret (3–4 sentences):**
Was our experiment ($n=500$) well-powered to detect the true 6pp lift?
From the table and chart, what is the minimum sample size per group
needed to reliably detect a 6pp lift at 80% power?
What happens to the MDE when you quadruple (not double) the sample size?
Does the relationship look linear on the chart?


**Answer:** TODO


## Question 14 — Computing Required Sample Size  *(4 points)*

The streaming service's product team says:
*"We only care about rolling out the new algorithm if it lifts completion
by at least 3 percentage points."*

Compute the **minimum sample size per group** needed to detect a 3pp lift
at 80% power and $\alpha = 0.05$. Use the formula:

$$n \;\geq\; \frac{(z_{1-\alpha/2} + z_{1-\beta})^2 \;\cdot\; 2\sigma^2}{\delta^2}$$


In [ ]:
# TODO: Compute the required n per group to detect delta = 3pp lift
# at alpha=0.05, power=0.80, baseline=0.35

delta   = 0.03      # MDE = 3pp
sigma   = # TODO: sqrt(BASELINE_RATE * (1 - BASELINE_RATE))
z_alpha = stats.norm.ppf(1 - ALPHA / 2)
z_power = stats.norm.ppf(0.80)

n_required = # TODO: apply the formula above, round up

print(f'Required n per group to detect {delta:.0%} lift: {n_required}')
print(f'Total users needed (both groups): {2 * n_required}')
print()
print(f'Verification — MDE at n={n_required}: {compute_mde(n_required):.2%}')
print(f'(should be approximately {delta:.0%})')


**In one sentence:** If the service gets 10,000 unique users per day,
how many days would the experiment need to run to reach the required sample size?


**Answer:** TODO


---
# Part 6: Break the Estimator — Peeking and Attrition

> *"The best way to understand an assumption is to violate it on purpose."*

You will now **deliberately violate two key assumptions** of A/B testing:
(1) pre-specified stopping (no peeking), and (2) complete outcome observation
(no attrition). Both inflate or deflate your estimates in predictable ways.


## Question 15 — The Peeking Problem  *(7 points)*

### Write your prior first

You are about to simulate what happens to the **false positive rate** when
an analyst checks the data repeatedly and stops the experiment early whenever
$p < 0.05$. The true effect is zero (pure noise).

**Before running the code:** If you run 1,000 experiments on pure noise and
peek at the data 10 times each, what false positive rate do you expect?
Why is it higher than 5%?


**Prior:** TODO


In [ ]:
def peeking_fpr(n_total=1000, looks=10, sims=2000, seed=42):
    """Simulate false positive rate under peeking with no true effect."""
    rng_p = np.random.default_rng(seed)
    checkpoints = np.unique(np.linspace(50, n_total, looks, dtype=int))
    false_positives = 0
    for _ in range(sims):
        data   = rng_p.binomial(1, BASELINE_RATE, n_total * 2)
        c_all  = data[:n_total]
        t_all  = data[n_total:]
        for cp in checkpoints:
            c, t = c_all[:cp], t_all[:cp]
            if len(c) < 2:
                continue
            se_p = np.sqrt(t.mean()*(1-t.mean())/len(t) +
                           c.mean()*(1-c.mean())/len(c))
            if se_p == 0:
                continue
            z_p = (t.mean() - c.mean()) / se_p
            if 2 * (1 - stats.norm.cdf(abs(z_p))) < ALPHA:
                false_positives += 1
                break
    return false_positives / sims

# TODO: Compute FPR for looks in {1, 2, 5, 10, 20} and store in a dict
look_counts = [1, 2, 5, 10, 20]
fpr_results = {}   # key: number of looks, value: false positive rate

for k in look_counts:
    fpr_results[k] = # TODO: call peeking_fpr with looks=k
    print(f'Looks = {k:>2}:  FPR = {fpr_results[k]:.1%}')


In [ ]:
# TODO: Plot FPR vs number of looks.
# Requirements:
#   - Bar chart or line plot with number of looks on x-axis
#   - FPR (%) on y-axis
#   - Red dashed horizontal line at 5% (nominal level)
#   - Label axes and add a title
#   - Save to exports/peeking_fpr.png

# YOUR CODE HERE

plt.savefig('exports/peeking_fpr.png', dpi=150, bbox_inches='tight')
plt.show()


**Interpret (3–4 sentences):**
At 10 peeks, what is the approximate false positive rate?
A company runs 200 A/B tests per year, each peeked 10 times.
How many false positives should they expect per year, compared to a company
that never peeks?
Name one method that allows interim analysis without inflating the false positive rate.


**Answer:** TODO


## Question 16 — Attrition Bias  *(6 points)*

**Attrition** occurs when some users drop out of the experiment before
their outcome is measured. It is harmless if dropout is random.
It is dangerous if dropout is **related to the outcome** — because then
the remaining sample is no longer representative of the original population.

### Scenario
In the streaming experiment, imagine that 20% of **control group** users
who would have **dropped off** (y=0) cancelled their subscription mid-experiment
and could not be measured. This is **differential attrition**: it affects
the control group more, and it removes the users *least likely* to complete sessions.

### Write your prior first

Before running the code: in which direction do you expect the biased estimate
to move relative to the true lift? Will it be higher or lower than 6pp? Why?


**Prior:** TODO


In [ ]:
# ── Introduce attrition in the control group ─────────────────────────────────
ATTRITION_RATE = 0.20   # 20% of control non-completers drop out

rng_att = np.random.default_rng(777)
df_att  = df.copy()

# Identify control users who did NOT complete (y=0) — these are the ones who
# cancel their subscription and become unobserved
attrition_candidates = df_att[(df_att.treat == 0) & (df_att.y == 0)].index
drop_mask = rng_att.random(len(attrition_candidates)) < ATTRITION_RATE
drop_idx  = attrition_candidates[drop_mask]

df_att = df_att.drop(index=drop_idx).reset_index(drop=True)

print(f'Original dataset:   {len(df)} rows')
print(f'After attrition:    {len(df_att)} rows')
print(f'Users dropped:      {len(drop_idx)}')
print(f'Control n (original):  {N_per_group}')
print(f'Control n (remaining): {(df_att.treat==0).sum()}')

# TODO: Compute the biased ATE estimate on df_att using the same
#       difference-in-means formula as Question 7

rate_c_att = # TODO: control completion rate in the attrite-d sample
rate_t_att = # TODO: treatment completion rate (unchanged)
tau_biased  = # TODO

print()
print(f'True lift                     : {TRUE_LIFT:.3%}')
print(f'Clean estimate (no attrition) : {tau_hat:.3%}')
print(f'Biased estimate (attrition)   : {tau_biased:.3%}')
print(f'Bias                          : {tau_biased - TRUE_LIFT:.3%}')


**Interpret (3–4 sentences):**
Was the bias in the direction you predicted?
Explain *mechanically* why removing low-outcome control users inflates the
estimate. What would an analyst need to do to detect attrition bias in
a real experiment? Name one diagnostic check.


**Answer:** TODO


---
# Part 7: Non-compliance — ITT and LATE

### Scenario

The streaming service discovers that **25% of users assigned to the new
algorithm** actually experienced a technical bug and heard the old algorithm
instead. These users were **assigned** to treatment but never **received** it.

We now have two indicators:

| Variable | Meaning | Randomized? |
|---|---|---|
| `treat` (= $Z$) | Assigned to new algorithm | ✅ Yes |
| `received` (= $D$) | Actually heard new algorithm | ❌ No — affected by the bug |


## Question 17 — Write Your Prior for ITT vs. LATE  *(2 points)*

**(a)** With 25% non-compliance, do you expect the ITT to be higher or
lower than the clean RCT estimate? By how much approximately?

**(b)** The LATE is defined as ITT / Compliance Rate.
What do you expect the LATE to be approximately, and why is it larger than the ITT?


**Prior (a):** TODO

**Prior (b):** TODO


## Question 18 — Compute ITT, Naive Estimate, and LATE  *(10 points)*


In [ ]:
# ── Introduce non-compliance ─────────────────────────────────────────────────
COMPLIANCE_RATE = 0.75   # 75% of assigned-treatment users actually receive it

rng_nc = np.random.default_rng(2025)
df_nc  = df.copy()

treat_idx      = df_nc[df_nc.treat == 1].index
non_comply_mask = rng_nc.random(len(treat_idx)) > COMPLIANCE_RATE
non_comply_idx  = treat_idx[non_comply_mask]

df_nc['received']              = df_nc['treat'].copy()
df_nc.loc[non_comply_idx, 'received'] = 0

print('Assignment vs actual receipt:')
print(df_nc.groupby(['treat','received']).size().rename('n').reset_index().to_string(index=False))
print(f'\nActual compliance rate: {df_nc.loc[df_nc.treat==1, "received"].mean():.1%}')
print()

# ── (a) ITT: compare by ASSIGNMENT (treat), not by received ──────────────────
ITT = # TODO: mean(y | treat=1) - mean(y | treat=0)

# ── (b) Naive: compare by RECEIPT (received) — WRONG approach ────────────────
naive_nc = # TODO: mean(y | received=1) - mean(y | received=0)

# ── (c) First stage: effect of assignment on receipt ─────────────────────────
first_stage = # TODO: mean(received | treat=1) - mean(received | treat=0)

# ── (d) LATE = ITT / first stage ─────────────────────────────────────────────
LATE = # TODO

print(f'True lift (known)              : {TRUE_LIFT:.3%}')
print(f'Clean RCT estimate             : {tau_hat:.3%}')
print(f'ITT (assignment-based)         : {ITT:.3%}')
print(f'Naive (receipt-based, biased)  : {naive_nc:.3%}')
print(f'First stage (compliance rate)  : {first_stage:.3%}')
print(f'LATE (for compliers only)      : {LATE:.3%}')


**Interpret (4–5 sentences):**
Explain in plain English what the ITT and LATE each measure.
Which is larger here — ITT or LATE? Is that what you expected?
Explain why comparing users by receipt (`received`) rather than assignment (`treat`)
re-introduces selection bias. In a business context, when would a manager prefer
the ITT over the LATE, and when would they prefer the LATE?


**Answer:** TODO


---
## Final Export


In [ ]:
# Save clean data
df.to_csv('data_clean/ab_test_clean.csv', index=False)
print('Clean data saved to data_clean/ab_test_clean.csv')
print('\nExports folder contents:')
for f in sorted(os.listdir('exports')):
    print(' ', f)


---
## Submission Checklist

Before uploading to Blackboard, confirm:

- [ ] Notebook runs **top → bottom without errors**
- [ ] Every **TODO** replaced with a real answer
- [ ] All three figures saved in `exports/`:
  - `ab_result.png`
  - `power_curve.png`
  - `peeking_fpr.png`
- [ ] Written answers are in **Markdown cells** (not code comments)
- [ ] Saved as **`.ipynb`** and uploaded to Blackboard Ultra

---
### Point Summary

| Part | Questions | Points |
|---|---|---|
| Part 1 — Conceptual | Q1–Q4 | 18 |
| Part 2 — Simulate | Q5–Q6 | 7 |
| Part 3 — ATE Estimation | Q7–Q8 | 17 |
| Part 4 — OLS & ANCOVA | Q9–Q11 | 19 |
| Part 5 — Power & MDE | Q12–Q14 | 14 |
| Part 6 — Break the Estimator | Q15–Q16 | 13 |
| Part 7 — ITT & LATE | Q17–Q18 | 12 |
| **Total** | | **100** |

---
### Further Reading (Optional)
- **Kohavi, Tang & Xu (2020)**, *Trustworthy Online Controlled Experiments* (O'Reilly)
- **Cunningham (2021)**, *Causal Inference: The Mixtape*, Ch. 4 — free at [mixtape.scunning.com](https://mixtape.scunning.com)
- **Simonsohn et al. (2014)**, *'p-curve: A Key to the File Drawer'*
